# 01 · Bronze Layer — Extraction (E)

This notebook extracts data from the WWI (OLTP) source to the bronze layer of the DW,
applying **different extraction strategies** depending on the table type:

| Tables | Strategy | Justification |
|---|---|---|
| Customers, CustomerCategories, BuyingGroups, People | **Differential snapshot** | No reliable temporal field |
| StockItems, StockGroups, StockItemStockGroups | **Differential snapshot** | Reference data |
| Suppliers, Colors, PackageTypes | **Differential snapshot** | Reference data |
| Cities, StateProvinces, Countries, DeliveryMethods | **Differential snapshot** | Reference data |
| Invoices, InvoiceLines | **Incremental by `invoice_date`** | Immutable creation date |
| Orders, OrderLines | **Incremental by `order_date`** | Immutable creation date |

---
> **⚠ Important:** Verify the exact schema and table names in your
> WWI instance before running (use cell 8 of `00_setup.ipynb`).
> By default, the standard WWI PostgreSQL schemas are assumed:
> `sales`, `warehouse`, `application`, `purchasing`.
> If your server uses only `public`, update `SRC_SCHEMA_*` below.

## 1. Imports and connections

In [21]:
import pandas as pd
import hashlib
from datetime import datetime, date, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

SRC_HOST     = os.getenv("SRC_HOST")
SRC_PORT     = int(os.getenv("SRC_PORT") or 5432)
SRC_DB       = os.getenv("SRC_DB")
SRC_USER     = os.getenv("SRC_USER")
SRC_PASSWORD = os.getenv("SRC_PASSWORD")

DWH_HOST     = os.getenv("DWH_HOST")
DWH_PORT     = int(os.getenv("DWH_PORT") or 5432)
DWH_DB       = os.getenv("DWH_DB")
DWH_USER     = os.getenv("DWH_USER")
DWH_PASSWORD = os.getenv("DWH_PASSWORD")

engine_src = create_engine(
    f"postgresql+psycopg2://{SRC_USER}:{SRC_PASSWORD}@{SRC_HOST}:{SRC_PORT}/{SRC_DB}"
)
engine_dwh = create_engine(
    f"postgresql+psycopg2://{DWH_USER}:{DWH_PASSWORD}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}"
)

print("✓ Engines created.")
print(f"  SRC: {SRC_USER}@{SRC_HOST}:{SRC_PORT}/{SRC_DB}")
print(f"  DWH: {DWH_USER}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}")

✓ Engines created.
  SRC: dss@postgres2.ipca.pt:5432/wwi
  DWH: postgres@localhost:5432/wwi_dw


## 2. Schema and table configuration by strategy

> **Adjust the schemas below** according to your WWI instance.
> Use `public` if all tables are in a single schema.

In [22]:
# Single schema in the IPCA WWI source — all tables are in 'public'
SCHEMA = os.getenv("SRC_SCHEMA_SALES", "public")

# ── Tables extracted by differential snapshot ─────────────────────────────────
SNAPSHOT_TABLES = [
    {"src_schema": SCHEMA, "src_table": "customers",          "bronze_table": "customers",          "pk": ["customerid"]},
    {"src_schema": SCHEMA, "src_table": "customercategories", "bronze_table": "customercategories", "pk": ["customercategoryid"]},
    {"src_schema": SCHEMA, "src_table": "buyinggroups",       "bronze_table": "buyinggroups",       "pk": ["buyinggroupid"]},
    {"src_schema": SCHEMA, "src_table": "people",             "bronze_table": "people",             "pk": ["personid"]},
    {"src_schema": SCHEMA, "src_table": "stockitems",         "bronze_table": "stockitems",         "pk": ["stockitemid"]},
    {"src_schema": SCHEMA, "src_table": "stockgroups",        "bronze_table": "stockgroups",        "pk": ["stockgroupid"]},
    {"src_schema": SCHEMA, "src_table": "colors",             "bronze_table": "colors",             "pk": ["colorid"]},
    {"src_schema": SCHEMA, "src_table": "packagetypes",       "bronze_table": "packagetypes",       "pk": ["packagetypeid"]},
    {"src_schema": SCHEMA, "src_table": "cities",             "bronze_table": "cities",             "pk": ["cityid"]},
    {"src_schema": SCHEMA, "src_table": "stateprovinces",     "bronze_table": "stateprovinces",     "pk": ["stateprovinceid"]},
    {"src_schema": SCHEMA, "src_table": "countries",          "bronze_table": "countries",          "pk": ["countryid"]},
    {"src_schema": SCHEMA, "src_table": "deliverymethods",    "bronze_table": "deliverymethods",    "pk": ["deliverymethodid"]},
]

# ── Tables extracted incrementally by date ────────────────────────────────────
DATE_TABLES = [
    {"src_schema": SCHEMA, "src_table": "invoices",     "bronze_table": "invoices",     "watermark_col": "invoicedate"},
    {"src_schema": SCHEMA, "src_table": "invoicelines", "bronze_table": "invoicelines", "watermark_col": "invoicedate"},
    {"src_schema": SCHEMA, "src_table": "orders",       "bronze_table": "orders",       "watermark_col": "orderdate"},
    {"src_schema": SCHEMA, "src_table": "orderlines",   "bronze_table": "orderlines",   "watermark_col": "orderdate"},
]

print("Differential snapshot strategy:")
for t in SNAPSHOT_TABLES:
    print(f"  {t['src_schema']}.{t['src_table']:<25}  bronze.{t['bronze_table']}")

print("\nIncremental by date strategy:")
for t in DATE_TABLES:
    print(f"  {t['src_schema']}.{t['src_table']:<15}  bronze.{t['bronze_table']:<15}  watermark: {t['watermark_col']}")

Differential snapshot strategy:
  public.customers                  bronze.customers
  public.customercategories         bronze.customercategories
  public.buyinggroups               bronze.buyinggroups
  public.people                     bronze.people
  public.stockitems                 bronze.stockitems
  public.stockgroups                bronze.stockgroups
  public.colors                     bronze.colors
  public.packagetypes               bronze.packagetypes
  public.cities                     bronze.cities
  public.stateprovinces             bronze.stateprovinces
  public.countries                  bronze.countries
  public.deliverymethods            bronze.deliverymethods

Incremental by date strategy:
  public.invoices         bronze.invoices         watermark: invoicedate
  public.invoicelines     bronze.invoicelines     watermark: invoicedate
  public.orders           bronze.orders           watermark: orderdate
  public.orderlines       bronze.orderlines       watermark: ord

## 3. Control table

`bronze._load_control` records the history of each load — used by both strategies.

In [23]:
DDL_CONTROL = """
    CREATE TABLE IF NOT EXISTS bronze._load_control (
        table_name      VARCHAR(100)  NOT NULL,
        strategy        VARCHAR(20)   NOT NULL,
        snapshot_id     INT,
        watermark_date  DATE,
        loaded_at       TIMESTAMP     NOT NULL,
        rows_total      INT,
        rows_inserted   INT,
        rows_updated    INT,
        rows_deleted    INT,
        status          VARCHAR(20)   NOT NULL,
        PRIMARY KEY (table_name, loaded_at)
    );
"""

def table_exists(conn, schema: str, table: str) -> bool:
    sql = """
        SELECT EXISTS (
            SELECT 1 FROM information_schema.tables
            WHERE table_schema = :schema AND table_name = :tname
        )
    """
    return conn.execute(text(sql), {"schema": schema, "tname": table}).scalar()

with engine_dwh.begin() as conn:
    already_existed = table_exists(conn, "bronze", "_load_control")
    conn.execute(text(DDL_CONTROL))

if already_existed:
    print("✓ Table bronze._load_control already exists — skipped.")
else:
    print("✓ Table bronze._load_control created.")

✓ Table bronze._load_control already exists — skipped.


## 4. Bronze table creation

Snapshot tables include `_snapshot_id` and `_change_op`.  
Incremental tables include only `_loaded_at` and `_source`.

In [24]:
SNAP_META = """
    _snapshot_id  INT          NOT NULL,
    _loaded_at    TIMESTAMP    NOT NULL,
    _change_op    VARCHAR(10)  NOT NULL,
    _source       VARCHAR(100) NOT NULL
"""

DATE_META = """
    _loaded_at    TIMESTAMP    NOT NULL,
    _source       VARCHAR(100) NOT NULL
"""

DDL_BRONZE = {

"customers": f"""
    CREATE TABLE IF NOT EXISTS bronze.customers (
        customerid              INT           NOT NULL,
        customername            VARCHAR(100),
        billtocustomerid        INT,
        customercategoryid      INT,
        buyinggroupid           INT,
        primarycontactpersonid  INT,
        deliverymethodid        INT,
        deliverycityid          INT,
        isoncredithold          SMALLINT,
        phonenumber             VARCHAR(50),
        websiteurl              VARCHAR(250),
        {SNAP_META}
    );
""",

"customercategories": f"""
    CREATE TABLE IF NOT EXISTS bronze.customercategories (
        customercategoryid    INT          NOT NULL,
        customercategoryname  VARCHAR(50),
        {SNAP_META}
    );
""",

"buyinggroups": f"""
    CREATE TABLE IF NOT EXISTS bronze.buyinggroups (
        buyinggroupid    INT          NOT NULL,
        buyinggroupname  VARCHAR(50),
        {SNAP_META}
    );
""",

"people": f"""
    CREATE TABLE IF NOT EXISTS bronze.people (
        personid        INT           NOT NULL,
        fullname        VARCHAR(100),
        preferredname   VARCHAR(50),
        issalesperson   SMALLINT,
        isemployee      SMALLINT,
        phonenumber     VARCHAR(50),
        emailaddress    VARCHAR(255),
        {SNAP_META}
    );
""",

"stockitems": f"""
    CREATE TABLE IF NOT EXISTS bronze.stockitems (
        stockitemid      INT            NOT NULL,
        stockitemname    VARCHAR(100),
        supplierid       INT,
        colorid          INT,
        unitpackageid    INT,
        outerpackageid   INT,
        brand            VARCHAR(50),
        size             VARCHAR(50),
        leadtimedays     INT,
        quantityperouter INT,
        ischillerstock   SMALLINT,
        taxrate          NUMERIC(5,2),
        unitprice        NUMERIC(10,2),
        {SNAP_META}
    );
""",

"stockgroups": f"""
    CREATE TABLE IF NOT EXISTS bronze.stockgroups (
        stockgroupid    INT          NOT NULL,
        stockgroupname  VARCHAR(50),
        {SNAP_META}
    );
""",

"colors": f"""
    CREATE TABLE IF NOT EXISTS bronze.colors (
        colorid    INT          NOT NULL,
        colorname  VARCHAR(50),
        {SNAP_META}
    );
""",

"packagetypes": f"""
    CREATE TABLE IF NOT EXISTS bronze.packagetypes (
        packagetypeid    INT          NOT NULL,
        packagetypename  VARCHAR(50),
        {SNAP_META}
    );
""",

"cities": f"""
    CREATE TABLE IF NOT EXISTS bronze.cities (
        cityid           INT           NOT NULL,
        cityname         VARCHAR(100),
        stateprovinceid  INT,
        {SNAP_META}
    );
""",

"stateprovinces": f"""
    CREATE TABLE IF NOT EXISTS bronze.stateprovinces (
        stateprovinceid    INT           NOT NULL,
        stateprovincename  VARCHAR(100),
        countryid          INT,
        salesterritory     VARCHAR(50),
        {SNAP_META}
    );
""",

"countries": f"""
    CREATE TABLE IF NOT EXISTS bronze.countries (
        countryid    INT           NOT NULL,
        ountryname   VARCHAR(100),
        {SNAP_META}
    );
""",

"deliverymethods": f"""
    CREATE TABLE IF NOT EXISTS bronze.deliverymethods (
        deliverymethodid    INT          NOT NULL,
        deliverymethodname  VARCHAR(50),
        {SNAP_META}
    );
""",

"invoices": f"""
    CREATE TABLE IF NOT EXISTS bronze.invoices (
        invoiceid            INT   NOT NULL,
        customerid           INT,
        billtocustomerid     INT,
        orderid              INT,
        deliverymethodid     INT,
        salespersonpersonid  INT,
        invoicedate          DATE,
        {DATE_META}
    );
""",

"invoicelines": f"""
    CREATE TABLE IF NOT EXISTS bronze.invoicelines (
        invoicelineid  INT            NOT NULL,
        invoiceid      INT,
        stockitemid    INT,
        quantity       INT,
        unitprice      NUMERIC(18,2),
        taxamount      NUMERIC(18,2),
        lineprofit     NUMERIC(18,2),
        extendedprice  NUMERIC(18,2),
        {DATE_META}
    );
""",

"orders": f"""
    CREATE TABLE IF NOT EXISTS bronze.orders (
        orderid               INT   NOT NULL,
        customerid            INT,
        salespersonpersonid   INT,
        pickedbypersonid      INT,
        contactpersonid       INT,
        orderdate             DATE,
        expecteddeliverydate  DATE,
        {DATE_META}
    );
""",

"orderlines": f"""
    CREATE TABLE IF NOT EXISTS bronze.orderlines (
        orderlineid     INT            NOT NULL,
        orderid         INT,
        stockitemid     INT,
        quantity        INT,
        pickedquantity  INT,
        unitprice       NUMERIC(18,2),
        taxrate         NUMERIC(5,2),
        {DATE_META}
    );
""",
}

with engine_dwh.begin() as conn:
    created, skipped = [], []
    for name, ddl in DDL_BRONZE.items():
        existed = table_exists(conn, "bronze", name)
        conn.execute(text(ddl))
        (skipped if existed else created).append(name)

if created:
    print("Tables created:")
    for name in created:
        print(f"  ✓ bronze.{name}")
if skipped:
    print("Tables already exist — skipped:")
    for name in skipped:
        print(f"    bronze.{name}")

Tables already exist — skipped:
    bronze.customers
    bronze.customercategories
    bronze.buyinggroups
    bronze.people
    bronze.stockitems
    bronze.stockgroups
    bronze.colors
    bronze.packagetypes
    bronze.cities
    bronze.stateprovinces
    bronze.countries
    bronze.deliverymethods
    bronze.invoices
    bronze.invoicelines
    bronze.orders
    bronze.orderlines


## 5. Functions — Differential snapshot strategy

In [25]:
def get_next_snapshot_id(table_name: str) -> int:
    sql = """
        SELECT COALESCE(MAX(snapshot_id), 0) + 1
        FROM   bronze._load_control
        WHERE  table_name = :tname
    """
    with engine_dwh.connect() as conn:
        result = conn.execute(text(sql), {"tname": table_name}).scalar()
    return int(result) if result is not None else 1


def get_last_snapshot(table_name: str, pk: list) -> pd.DataFrame:
    """Reads the most recent snapshot from bronze (only active records)."""
    sql = f"""
        SELECT *
        FROM   bronze.{table_name}
        WHERE  _snapshot_id = (SELECT MAX(_snapshot_id) FROM bronze.{table_name})
          AND  _change_op  != 'DELETE'
    """
    try:
        with engine_dwh.connect() as conn:
            df = pd.read_sql(text(sql), conn)
        meta = ["_snapshot_id", "_loaded_at", "_change_op", "_source"]
        return df.drop(columns=[c for c in meta if c in df.columns])
    except Exception:
        return pd.DataFrame()


def compute_row_hash(df: pd.DataFrame, cols: list) -> pd.Series:
    return df[cols].astype(str).apply(
        lambda row: hashlib.md5("|".join(row).encode()).hexdigest(), axis=1
    )


def diff_snapshots(df_new: pd.DataFrame, df_prev: pd.DataFrame, pk: list) -> pd.DataFrame:
    """Compares two snapshots and returns only changed rows with _change_op."""
    df_new = df_new.copy().set_index(pk)

    if df_prev.empty:
        result = df_new.reset_index()
        result["_change_op"] = "INSERT"
        return result

    df_prev = df_prev.copy().set_index(pk)
    common_cols = [c for c in df_new.columns if c in df_prev.columns]

    hash_new  = compute_row_hash(df_new.reset_index(),  pk + common_cols)
    hash_prev = compute_row_hash(df_prev.reset_index(), pk + common_cols)
    hash_new.index  = df_new.index
    hash_prev.index = df_prev.index

    changed_rows = []

    new_keys = df_new.index.difference(df_prev.index)
    if len(new_keys):
        ins = df_new.loc[new_keys].reset_index()
        ins["_change_op"] = "INSERT"
        changed_rows.append(ins)

    common_keys  = df_new.index.intersection(df_prev.index)
    updated_keys = common_keys[hash_new.loc[common_keys] != hash_prev.loc[common_keys]]
    if len(updated_keys):
        upd = df_new.loc[updated_keys].reset_index()
        upd["_change_op"] = "UPDATE"
        changed_rows.append(upd)

    deleted_keys = df_prev.index.difference(df_new.index)
    if len(deleted_keys):
        dlt = df_prev.loc[deleted_keys].reset_index().reindex(columns=df_new.reset_index().columns)
        dlt["_change_op"] = "DELETE"
        changed_rows.append(dlt)

    return pd.concat(changed_rows, ignore_index=True) if changed_rows else pd.DataFrame()


print("✓ Snapshot functions defined.")

✓ Snapshot functions defined.


## 6. Functions — Incremental by date strategy

In [26]:
DATE_ZERO = date(1900, 1, 1)

def get_watermark_date(bronze_table: str, date_col: str) -> date:
    sql = f"SELECT MAX({date_col}) FROM bronze.{bronze_table}"
    try:
        with engine_dwh.connect() as conn:
            result = conn.execute(text(sql)).scalar()
        return result if result is not None else DATE_ZERO
    except Exception:
        return DATE_ZERO


def extract_invoices(watermark: date, schema: str) -> pd.DataFrame:
    sql = f"""
        SELECT invoiceid, customerid, billtocustomerid, orderid,
               deliverymethodid, salespersonpersonid, invoicedate
        FROM   {schema}.invoices
        WHERE  invoicedate > :wm
        ORDER  BY invoicedate
    """
    with engine_src.connect() as conn:
        return pd.read_sql(text(sql), conn, params={"wm": watermark})


def extract_invoicelines(new_invoice_ids: list, schema: str) -> pd.DataFrame:
    if not new_invoice_ids:
        return pd.DataFrame()
    sql = f"""
        SELECT invoicelineid, invoiceid, stockitemid,
               quantity, unitprice, taxamount, lineprofit, extendedprice
        FROM   {schema}.invoicelines
        WHERE  invoiceid = ANY(:ids)
    """
    with engine_src.connect() as conn:
        return pd.read_sql(text(sql), conn, params={"ids": new_invoice_ids})  # type: ignore[arg-type]


def extract_orders(watermark: date, schema: str) -> pd.DataFrame:
    sql = f"""
        SELECT orderid, customerid, salespersonpersonid,
               pickedbypersonid, contactpersonid,
               orderdate, expecteddeliverydate
        FROM   {schema}.orders
        WHERE  orderdate > :wm
        ORDER  BY orderdate
    """
    with engine_src.connect() as conn:
        return pd.read_sql(text(sql), conn, params={"wm": watermark})


def extract_orderlines(new_order_ids: list, schema: str) -> pd.DataFrame:
    if not new_order_ids:
        return pd.DataFrame()
    sql = f"""
        SELECT orderlineid, orderid, stockitemid,
               quantity, pickedquantity, unitprice, taxrate
        FROM   {schema}.orderlines
        WHERE  orderid = ANY(:ids)
    """
    with engine_src.connect() as conn:
        return pd.read_sql(text(sql), conn, params={"ids": new_order_ids})  # type: ignore[arg-type]


print("✓ Date-based extraction functions defined.")

✓ Date-based extraction functions defined.


## 7. Control logging function

In [27]:
def log_control(table_name, strategy, snapshot_id=None, watermark_date=None,
                rows_total=0, rows_inserted=0, rows_updated=0, rows_deleted=0,
                status="OK"):
    sql = """
        INSERT INTO bronze._load_control
          (table_name, strategy, snapshot_id, watermark_date, loaded_at,
           rows_total, rows_inserted, rows_updated, rows_deleted, status)
        VALUES (:tname, :strat, :snap, :wm, :ts,
                :total, :ins, :upd, :del, :status)
    """
    with engine_dwh.begin() as conn:
        conn.execute(text(sql), {
            "tname": table_name, "strat": strategy, "snap": snapshot_id,
            "wm": watermark_date, "ts": datetime.now(timezone.utc).replace(tzinfo=None),
            "total": rows_total, "ins": rows_inserted, "upd": rows_updated,
            "del": rows_deleted, "status": status,
        })

print("✓ Control function defined.")

✓ Control function defined.


## 8. Execution — Snapshot Extraction (reference tables)

In [28]:
loaded_at = datetime.now(timezone.utc).replace(tzinfo=None)

for cfg in SNAPSHOT_TABLES:
    src   = f"{cfg['src_schema']}.{cfg['src_table']}"
    brnz  = cfg["bronze_table"]
    pk    = cfg["pk"]
    snap_id = get_next_snapshot_id(brnz)

    try:
        # 1. Read all current data from source
        with engine_src.connect() as conn:
            df_new = pd.read_sql(text(f"SELECT * FROM {src}"), conn)

        # 2. Read previous snapshot from bronze
        df_prev = get_last_snapshot(brnz, pk)

        # 3. Compute differences
        df_diff = diff_snapshots(df_new, df_prev, pk)

        n_ins, n_upd, n_del = 0, 0, 0

        if not df_diff.empty:
            df_diff["_snapshot_id"] = snap_id
            df_diff["_loaded_at"]   = loaded_at
            df_diff["_source"]      = src

            # Select only columns that exist in the bronze table
            with engine_dwh.connect() as conn:
                cols_sql = """
                    SELECT column_name FROM information_schema.columns
                    WHERE table_schema = 'bronze' AND table_name = :tname
                    ORDER BY ordinal_position
                """
                bronze_cols = [r[0] for r in conn.execute(text(cols_sql), {"tname": brnz}).fetchall()]

            df_to_insert = df_diff[[c for c in bronze_cols if c in df_diff.columns]]

            n_ins = len(df_to_insert[df_to_insert["_change_op"] == "INSERT"])
            n_upd = len(df_to_insert[df_to_insert["_change_op"] == "UPDATE"])
            n_del = len(df_to_insert[df_to_insert["_change_op"] == "DELETE"])

            with engine_dwh.begin() as conn:
                df_to_insert.to_sql(brnz, conn, schema="bronze",
                                    if_exists="append", index=False)

        log_control(brnz, "snapshot", snapshot_id=snap_id,
                    rows_total=len(df_new), rows_inserted=n_ins,
                    rows_updated=n_upd, rows_deleted=n_del)

        total_changes = n_ins + n_upd + n_del
        print(f"✓ bronze.{brnz:<30}  snap={snap_id}  total={len(df_new):>6}  "
              f"ins={n_ins}  upd={n_upd}  del={n_del}  "
              f"{'(no changes)' if total_changes == 0 else ''}")

    except Exception as e:
        log_control(brnz, "snapshot", snapshot_id=snap_id, status="ERROR")
        print(f"✗ bronze.{brnz}  ERROR: {e}")

✓ bronze.customers                       snap=5  total=   663  ins=0  upd=0  del=0  (no changes)
✓ bronze.customercategories              snap=5  total=     8  ins=0  upd=0  del=0  (no changes)
✓ bronze.buyinggroups                    snap=5  total=     2  ins=0  upd=0  del=0  (no changes)
✓ bronze.people                          snap=5  total=  1111  ins=0  upd=0  del=0  (no changes)
✓ bronze.stockitems                      snap=5  total=   227  ins=0  upd=0  del=0  (no changes)
✓ bronze.stockgroups                     snap=5  total=    10  ins=0  upd=0  del=0  (no changes)
✓ bronze.colors                          snap=5  total=    36  ins=0  upd=0  del=0  (no changes)
✓ bronze.packagetypes                    snap=5  total=    14  ins=0  upd=0  del=0  (no changes)
✓ bronze.cities                          snap=5  total= 37940  ins=0  upd=0  del=0  (no changes)
✓ bronze.stateprovinces                  snap=5  total=    53  ins=0  upd=0  del=0  (no changes)
✓ bronze.countries            

## 9. Execution — Incremental Extraction (transactional tables)

In [29]:
# ── Invoices + InvoiceLines ───────────────────────────────────────────────────
wm_invoices = get_watermark_date("invoices", "invoicedate")
print(f"Watermark invoices: {wm_invoices}")

df_inv = extract_invoices(wm_invoices, SCHEMA)
print(f"  New invoices:      {len(df_inv)}")

if not df_inv.empty:
    df_inv["_loaded_at"] = loaded_at
    df_inv["_source"]    = f"{SCHEMA}.invoices"
    with engine_dwh.begin() as conn:
        df_inv.to_sql("invoices", conn, schema="bronze", if_exists="append", index=False)

    new_invoice_ids = df_inv["invoiceid"].tolist()
    df_il = extract_invoicelines(new_invoice_ids, SCHEMA)
    print(f"  New invoicelines:  {len(df_il)}")

    if not df_il.empty:
        df_il["_loaded_at"] = loaded_at
        df_il["_source"]    = f"{SCHEMA}.invoicelines"
        with engine_dwh.begin() as conn:
            df_il.to_sql("invoicelines", conn, schema="bronze", if_exists="append", index=False)

    max_inv_date = df_inv["invoicedate"].max()
    log_control("invoices",     "incremental", watermark_date=max_inv_date, rows_inserted=len(df_inv))
    log_control("invoicelines", "incremental", watermark_date=max_inv_date,
                rows_inserted=len(df_il) if not df_il.empty else 0)

print()

# ── Orders + OrderLines ───────────────────────────────────────────────────────
wm_orders = get_watermark_date("orders", "orderdate")
print(f"Watermark orders: {wm_orders}")

df_ord = extract_orders(wm_orders, SCHEMA)
print(f"  New orders:      {len(df_ord)}")

if not df_ord.empty:
    df_ord["_loaded_at"] = loaded_at
    df_ord["_source"]    = f"{SCHEMA}.orders"
    with engine_dwh.begin() as conn:
        df_ord.to_sql("orders", conn, schema="bronze", if_exists="append", index=False)

    new_order_ids = df_ord["orderid"].tolist()
    df_ol = extract_orderlines(new_order_ids, SCHEMA)
    print(f"  New orderlines:   {len(df_ol)}")

    if not df_ol.empty:
        df_ol["_loaded_at"] = loaded_at
        df_ol["_source"]    = f"{SCHEMA}.orderlines"
        with engine_dwh.begin() as conn:
            df_ol.to_sql("orderlines", conn, schema="bronze", if_exists="append", index=False)

    max_ord_date = df_ord["orderdate"].max()
    log_control("orders",     "incremental", watermark_date=max_ord_date, rows_inserted=len(df_ord))
    log_control("orderlines", "incremental", watermark_date=max_ord_date,
                rows_inserted=len(df_ol) if not df_ol.empty else 0)

print("\n✓ Incremental extraction completed.")

Watermark invoices: 2020-05-31
  New invoices:      0

Watermark orders: 2020-05-31
  New orders:      0

✓ Incremental extraction completed.


## 10. Summary — Bronze row counts

In [30]:
import pandas as pd

bronze_tables = [
    "customers", "customercategories", "buyinggroups", "people",
    "stockitems", "stockgroups", "colors", "packagetypes",
    "cities", "stateprovinces", "countries", "deliverymethods",
    "invoices", "invoicelines", "orders", "orderlines",
]

rows = []
with engine_dwh.connect() as conn:
    for t in bronze_tables:
        try:
            cnt = conn.execute(text(f"SELECT COUNT(*) FROM bronze.{t}")).scalar()
            rows.append({"table": f"bronze.{t}", "records": cnt})
        except Exception:
            rows.append({"table": f"bronze.{t}", "records": "ERROR"})

df_summary = pd.DataFrame(rows)
print("Bronze row counts:")
print(df_summary.to_string(index=False))

print("\n✓ Bronze completed. Next step: run 02_silver.ipynb")

Bronze row counts:
                    table  records
         bronze.customers      663
bronze.customercategories        8
      bronze.buyinggroups        2
            bronze.people     1111
        bronze.stockitems      227
       bronze.stockgroups       10
            bronze.colors       36
      bronze.packagetypes       14
            bronze.cities    37940
    bronze.stateprovinces       53
         bronze.countries      190
   bronze.deliverymethods       10
          bronze.invoices    70510
      bronze.invoicelines   228265
            bronze.orders    73595
        bronze.orderlines   231412

✓ Bronze completed. Next step: run 02_silver.ipynb
